In [1]:
import numpy as np
import pandas as pd
import json
import os
from constants import *
from run import *
from models import *
from sim import *

import torch
import gpytorch

from scipy.stats import multivariate_normal

/Users/juar705/miniconda3/envs/bacterai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#import AA file for use

csv_file_path = "/Users/juar705/Downloads/mock_data.csv"
df = pd.read_csv(csv_file_path)
truncated_df = df.head(1500)
del truncated_df['environment']

In [ ]:
'''
seperate into X_train and y_train sets
    X_train will be the amino acid columns 
    y_train will be the growth column
'''

AA_columns = AA_SHORT
growth_columns = 'growth'

X_train = truncated_df[AA_columns].to_numpy()
y_train = truncated_df[growth_columns].to_numpy()

In [ ]:
"""
Loads model and likelihood state dicts from disk inside the function, reconstructing the model and likelihood everytime the function is called.
Repeats the loading and reconstruction of the models when this is not needed. It is more efficient and cleaner to let this function handle only
samples and predictions, while the model and likelihood are loaded and initialized in the main script or a separate initialization function.

def sample_GP(model, likelihood, X, n_samples=1):
    
    # Convert data to tensor
    train_x = torch.tensor(X_train, dtype=torch.float)
    train_y = torch.tensor(y_train, dtype=torch.float)
    
    test_x = torch.tensor(X, dtype=torch.float32)
    
    # Initialize likelihood and model method
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model = gpr.ExactGPModel(train_x, train_y, likelihood)
    
    # Load the state dictionaries from previously saved files
    model.load_state_dict(torch.load('/Users/juar705/BacterAI_pnnl/mockrun/Round1/gpr_model/gpr_model.pth'))
    likelihood.load_state_dict(torch.load('/Users/juar705/BacterAI_pnnl/mockrun/Round1/gpr_model/gpr_likelihood.pth'))
    
    # Set model and likelihood to evaluation mode
    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(test_x))
     
    # Get the mean and covariance
    mean = observed_pred.mean.numpy()
    cov = observed_pred.covariance_matrix.numpy()
    
    # Sample from the multivariate normal distribution
    samples = np.atleast_1d(multivariate_normal.rvs(mean, cov, size=n_samples))
    variances = np.diag(cov)
    
    return samples, variances

"""

"\ndef sample_GP(model, likelihood, X, n_samples=1):\n    \n    # Convert data to tensor\n    train_x = torch.tensor(X_train, dtype=torch.float)\n    train_y = torch.tensor(y_train, dtype=torch.float)\n    \n    test_x = torch.tensor(X, dtype=torch.float32)\n    \n    # Initialize likelihood and model method\n    likelihood = gpytorch.likelihoods.GaussianLikelihood()\n    model = gpr.ExactGPModel(train_x, train_y, likelihood)\n    \n    # Load the state dictionaries from previously saved files\n    model.load_state_dict(torch.load('/Users/juar705/BacterAI_pnnl/mockrun/Round1/gpr_model/gpr_model.pth'))\n    likelihood.load_state_dict(torch.load('/Users/juar705/BacterAI_pnnl/mockrun/Round1/gpr_model/gpr_likelihood.pth'))\n    \n    # Set model and likelihood to evaluation mode\n    model.eval()\n    likelihood.eval()\n\n    with torch.no_grad(), gpytorch.settings.fast_pred_var():\n        observed_pred = likelihood(model(test_x))\n     \n    # Get the mean and covariance\n    mean = obse

In [5]:
# uses already loaded model and likelihood to make predictions for new data
def sample_GP(model, likelihood, X, n_samples=1):
    
    test_x = torch.tensor(X, dtype=torch.float32)
    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(test_x))

    mean = observed_pred.mean.numpy()
    cov = observed_pred.covariance_matrix.numpy()

    samples = np.atleast_1d(multivariate_normal.rvs(mean, cov, size=n_samples))
    variances = np.diag(cov)
    return samples, variances

In [6]:
#changes in load_trained_models loads trained GPR model and likelihood objects from disk
# reconstructs the model and likelihood using trained data, loads their weights from files and returns an object of GPRModel class with loaded components
#call this once to get your trained model and likelihood ready for inference
class GPRModel(Model):
    def __init__(self, model_path):
        self.model = []
        self.likelihood = []
        self.model_path = model_path
        self.is_trained = False
        super().__init__(self, ModelType.GPR)
        
    #this method loads the trained models and likelihoods from the specified path, edited in. not in the main Models.py script.
    @classmethod
    def load_trained_models(cls, models_path, X_train=None, y_train=None):
        obj = cls(models_path)

        # Convert training data to tensors for model reconstruction
        train_x = torch.tensor(X_train, dtype=torch.float)
        train_y = torch.tensor(y_train, dtype=torch.float)

        for filename in os.listdir(models_path):
            if "model" in filename:
                # Reconstruct model and load state dict
                model_instance = gpr.ExactGPModel(train_x, train_y, gpytorch.likelihoods.GaussianLikelihood())
                state_dict = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE))
                model_instance.load_state_dict(state_dict)
                obj.model.append(model_instance)
            if "likelihood" in filename:
                likelihood_instance = gpytorch.likelihoods.GaussianLikelihood()
                state_dict = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE))
                likelihood_instance.load_state_dict(state_dict)
                obj.likelihood.append(likelihood_instance)

        obj.is_trained = True
        return obj
    
    def check_path(self):
        if not os.path.exists(self.model_path):
            os.makedirs(self.model_path)

    def train(self, X_train, y_train, **kwargs):
        # X_trainR = robjects.r.matrix(
        #     X_train, nrow=X_train.shape[0], ncol=X_train.shape[1]
        # )
        # y_trainR = robjects.r.matrix(y_train, nrow=y_train.shape[0], ncol=1)
        # self.model = self.gpr_lib.train_new_GP(X_trainR, y_trainR)
        self.check_path()
        self.model, self.likelihood = gpr.train_new_GP(X_train, y_train, self.model_path, **kwargs)
        self.is_trained = True

    def evaluate(self, X, clip=True, n=1):
        # X_evalR = robjects.r.matrix(X, nrow=X.shape[0], ncol=X.shape[1])
        if not self.is_trained:
            raise Exception("GPR model needs to be trained before evaluating.")
        
        #removed gpr to obtain straight from the notebook rather than the file
        samples, variances  = sample_GP(self.model[0], self.likelihood[0], X, n)
        # Do we want to clip samples?
        if clip:
            samples = np.clip(samples, 0, 1)
        return samples, variances

In [7]:
with open('config.json', 'r') as file:
    config = json.load(file)
MODEL_TYPE = ModelType(0)   
NEW_ROUND_N = 1
EXPT_FOLDER = config["experiment_path"]
new_round_folder = os.path.join(EXPT_FOLDER, f"Round{NEW_ROUND_N}")
if MODEL_TYPE == ModelType.GPR:
    models_folder = os.path.join(new_round_folder, f"gpr_model")
    model = GPRModel(models_folder)
    model.train(X_train, y_train)      

Iter 1/100 | Train loss: 1.1878
Iter 2/100 | Train loss: 1.1246
Iter 3/100 | Train loss: 1.0702
Iter 4/100 | Train loss: 1.0184
Iter 5/100 | Train loss: 0.9697
Iter 6/100 | Train loss: 0.9224
Iter 7/100 | Train loss: 0.8776
Iter 8/100 | Train loss: 0.8218
Iter 9/100 | Train loss: 0.7720
Iter 10/100 | Train loss: 0.7168
Iter 11/100 | Train loss: 0.6684
Iter 12/100 | Train loss: 0.6129
Iter 13/100 | Train loss: 0.5643
Iter 14/100 | Train loss: 0.5165
Iter 15/100 | Train loss: 0.4670
Iter 16/100 | Train loss: 0.4178
Iter 17/100 | Train loss: 0.3760
Iter 18/100 | Train loss: 0.3360
Iter 19/100 | Train loss: 0.3065
Iter 20/100 | Train loss: 0.2664
Iter 21/100 | Train loss: 0.2464
Iter 22/100 | Train loss: 0.2190
Iter 23/100 | Train loss: 0.1980
Iter 24/100 | Train loss: 0.1772
Iter 25/100 | Train loss: 0.1679
Iter 26/100 | Train loss: 0.1558
Iter 27/100 | Train loss: 0.1512
Iter 28/100 | Train loss: 0.1506
Iter 29/100 | Train loss: 0.1438
Iter 30/100 | Train loss: 0.1538
Iter 31/100 | Train

In [ ]:
#Create an array of ones for down direction or 0 for up direction

n_ingredients = len(AA_SHORT)
batch_size = config["batch_size"]
DIRECTION = SimDirection(config["direction"])


def media_array(n_ingredients,direction):
    if DIRECTION == SimDirection.DOWN:
        media = np.ones(n_ingredients)
        direction = SimDirection.DOWN
    elif DIRECTION == SimDirection.UP:
        media = np.zeros(n_ingredients)
        direction = SimDirection.UP
    else:
        raise ValueError("Error") 
    return media

media = media_array(n_ingredients, DIRECTION)

Media Array:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [9]:
def make_batch(
    model,
    media,
    new_round_n,
    batch_size,
    sim_types,
    rollout_trajectories,
    threshold,
    timeout=60,
    unique=True,
    direction=SimDirection.DOWN,
    go_beyond_frontier=True,
    used_experiments=None,
    redo_experiments=None,
):
    """ Make a new BacterAI batch; the main function that calls the simulation loops. """
    sim_types=sim_types
    n_types = len(sim_types)
    n_exps = batch_size // n_types
    batch_set = used_experiments
    sub_batches = []
    all_metrics = {}
    for idx, sim_type in enumerate(sim_types):
        if idx == n_types - 1:
            n_exps = batch_size - sum([len(x) for x in sub_batches])
        print(idx, sim_type, batch_size, n_exps, sum([len(x) for x in sub_batches]))
        batch, batch_set, metrics = perform_simulations(
            model,
            media,
            n_exps,
            threshold,
            sim_type,
            direction,
            new_round_n,
            unique=unique,
            timeout=timeout,
            batch_set=batch_set,
            n_rollout_trajectories=rollout_trajectories,
            go_beyond_frontier=go_beyond_frontier,
        )
        sub_batches.append(batch)
        all_metrics[sim_type.name] = metrics

    batch = pd.concat([redo_experiments] + sub_batches, ignore_index=True)
    return batch, batch_set, all_metrics

In [10]:
trained_set = pd.DataFrame(np.hstack((X_train, y_train.reshape(-1,1))))
used_experiments = set(map(tuple,trained_set.to_numpy()))
batch_data= used_experiments

#set parameters needed for make batch function
sim_types=[SimType(2)]
rollout_trajectories=config["n_rollouts"]
threshold=config['grow_threshold']
timeout=60 * 15
unique=True 
direction = SimDirection(0)
go_beyond_frontier=config['beyond_frontier']

# Make batch the main function that calls to make all simulations from perform simulations function
batch, batch_set, all_metrics = make_batch(
    model=GPRModel.load_trained_models(models_folder, X_train=X_train, y_train=y_train),
    media=media,
    new_round_n=NEW_ROUND_N,
    batch_size=50,
    sim_types=sim_types,
    rollout_trajectories=rollout_trajectories,
    threshold=threshold,
    timeout=timeout,
    unique=unique,
    direction=direction,
    go_beyond_frontier=go_beyond_frontier,
    used_experiments=None,
    redo_experiments=None,)

0 SimType.ROLLOUT 50 50 0


Performing ROLLOUT Sims (DOWN):   0%|          | 0/50 [00:00<?, ?it/s]/Users/juar705/miniconda3/envs/bacterai/lib/python3.10/site-packages/scipy/stats/_multivariate.py:762: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  out = random_state.multivariate_normal(mean, cov, size)
Performing ROLLOUT Sims (DOWN) (2 loops):   2%|▏         | 1/50 [00:00<00:45,  1.08it/s]


	ADDED: [0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	ADDED: [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND


Performing ROLLOUT Sims (DOWN) (3 loops):   6%|▌         | 3/50 [00:01<00:19,  2.43it/s]


	ADDED: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	ADDED: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND


...


	ADDED: [0 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 0] - FRONTIER

	ADDED: [0 0 0 0 0 1 0 0 1 0 0 1 1 0 0 0 0 0 0 0] - BEYOND


Performing ROLLOUT Sims (DOWN) (49 loops):  96%|█████████▌| 48/50 [00:20<00:00,  2.53it/s]


	ADDED: [0 0 0 1 0 1 1 0 0 0 0 1 1 0 0 1 0 0 0 1] - FRONTIER

	ADDED: [0 0 0 1 0 0 1 0 0 0 0 1 1 0 0 1 0 0 0 1] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND


Performing ROLLOUT Sims (DOWN) (52 loops): 100%|██████████| 50/50 [00:21<00:00,  2.33it/s]


	ADDED: [0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER
perform_simulations function took 21455.18 ms


In [11]:
print(f"Batch:\n,{batch}")

Batch:
,    0  1  2  3  4  5  6  7  8  9  ...  17  18  19     type  direction  \
0   0  0  0  0  0  0  1  1  0  0  ...   0   0   0  ROLLOUT       DOWN   
1   0  0  0  0  0  0  0  1  0  0  ...   0   0   0  ROLLOUT       DOWN   
2   0  0  0  0  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
3   0  0  0  0  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
4   1  0  0  0  0  1  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
5   1  0  0  0  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
6   0  0  0  1  1  1  1  1  1  0  ...   0   0   0  ROLLOUT       DOWN   
7   0  0  0  1  1  1  1  1  0  0  ...   0   0   0  ROLLOUT       DOWN   
8   0  0  0  1  0  1  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
9   0  0  0  1  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
10  1  0  0  0  0  1  1  1  1  0  ...   0   0   0  ROLLOUT       DOWN   
11  1  0  0  0  0  1  1  1  1  0  ...   0   0   0  ROLLOUT       DOWN   
12  0  1  1  1  1  1  1  1  1  1  ...   0  

In [12]:
print(f"Batch Set:\n{batch_set}")

Batch Set:
{(np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int

In [13]:
print(f"Metrics:\n{all_metrics}")

Metrics:
{'ROLLOUT': {'k_history': [], 'count_history': [], 'k_avg': 'n/a', 'count_avg': 'n/a', 'total_loops_count': 52, 'time_to_finish_sec': 21.43}}
